# D1-06 Data quality checks and linking workflow

This notebook continues the Day 1 import story with a realistic foreground inventory file.

We use the bundled `lci-carbon-fiber.xlsx` asset to inspect unlinked exchanges, apply a migration, and write a clean foreground database linked to BAFU.


## Learning goals

- Import a realistic foreground Excel inventory with `bw2io.ExcelImporter`.
- Inspect statistics and unlinked exchanges after applying strategies.
- Distinguish technosphere and biosphere linking problems.
- Apply a migration file to repair common naming / mapping issues.
- Write a linked foreground database into the course project.


## Background references

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236
- Benitez, A., Wulf, C., de Palmenaer, A., Lengersdorf, M., Röding, T., Grube, T., et al. (2021). Ecological assessment of fuel cell electric vehicles with special focus on type IV carbon fiber hydrogen tank. *Journal of Cleaner Production, 278*, 123277. https://doi.org/10.1016/j.jclepro.2020.123277


## 1) Load the project and foreground asset

We assume `D1-04` has already imported the BAFU background database as `bafu-2025`.


In [ ]:
from pathlib import Path

import bw2data as bd
import bw2io as bi

bd.projects.set_current('barcelona-rlcia-2026')
background_name = 'bafu-2025'
foreground_name = 'carbon-fiber-demo'
asset_path = Path('tutorials/DAY 1/assets/lci-carbon-fiber.xlsx')

print('Current project:', bd.projects.current)
print('Foreground asset:', asset_path.resolve())
print('Foreground asset found:', asset_path.exists())
print('Background database available:', background_name in bd.databases)


## 2) Create an importer and apply strategies

The first pass usually does not solve every link. That is expected.
The goal is to inspect what remains unlinked after the standard strategy pipeline.


In [ ]:
if not asset_path.exists():
    print('Foreground asset not found.')
elif background_name not in bd.databases:
    print("Run D1-04 first so that 'bafu-2025' exists.")
else:
    importer = bi.ExcelImporter(str(asset_path))
    importer.apply_strategies()
    importer.match_database(background_name, fields=('name', 'reference product', 'location', 'unit'))
    importer.match_database('biosphere3', fields=('name', 'unit', 'categories'))
    stats_before = importer.statistics()
    print('Statistics before migration:', stats_before)
    print('Number of unlinked exchanges before migration:', len(getattr(importer, 'unlinked', [])))


## 3) Inspect the remaining unlinked exchanges

This is the point where we decide whether the issue is a technosphere match, a biosphere match, or a naming problem that requires a migration.


In [ ]:
if 'importer' in globals():
    technosphere_unlinked = [u for u in importer.unlinked if u.get('type') == 'technosphere']
    biosphere_unlinked = [u for u in importer.unlinked if u.get('type') == 'biosphere']

    print('Technosphere unlinked examples:')
    for item in technosphere_unlinked[:5]:
        print('-', item.get('name'), '|', item.get('reference product'), '|', item.get('location'))

    print('Biosphere unlinked examples:')
    for item in biosphere_unlinked[:5]:
        print('-', item.get('name'), '|', item.get('categories'))


## Checkpoint 1

Count the number of unlinked technosphere exchanges and the number of unlinked biosphere exchanges.
Which type seems more problematic in this file?


In [ ]:
# TODO


In [ ]:
if 'importer' in globals():
    technosphere_unlinked = [u for u in importer.unlinked if u.get('type') == 'technosphere']
    biosphere_unlinked = [u for u in importer.unlinked if u.get('type') == 'biosphere']
    print('Number of unlinked technosphere exchanges:', len(technosphere_unlinked))
    print('Number of unlinked biosphere exchanges:', len(biosphere_unlinked))


## 4) Apply a migration file

A migration file lets us translate known outdated or mismatched names before matching again.
The entries below are adapted from the older PSI Brightway teaching material and target the known issues in this asset.


In [ ]:
migration_name = 'carbon-fiber-demo-fixes'
migration_data = {
    'fields': ['name', 'reference product', 'location', 'categories'],
    'data': [
        (
            ('market for ethylene glycol', 'ethylene glycol', 'GLO', ''),
            {'location': 'RER'},
        ),
        (
            ('air separation, cryogenic', 'nitrogen, liquid', 'GLO', ''),
            {'name': 'industrial gases production, cryogenic air separation', 'location': 'RER'},
        ),
        (
            ('air separation, cryogenic', 'nitrogen, liquid', 'RER', ''),
            {'name': 'industrial gases production, cryogenic air separation', 'location': 'RER'},
        ),
        (
            ('Argon-40', '', '', ('air',)),
            {'name': 'Argon'},
        ),
    ],
}

if 'importer' in globals():
    if migration_name not in bi.migrations:
        bi.Migration(migration_name).write(data=migration_data, description='Fixes for the Day 1 carbon-fiber foreground import')
    importer.data = bi.strategies.migrate_exchanges(db=importer.data, migration=migration_name)
    importer.match_database(background_name, fields=('name', 'reference product', 'location', 'unit'))
    importer.match_database('biosphere3', fields=('name', 'unit', 'categories'))
    stats_after = importer.statistics()
    print('Statistics after migration:', stats_after)
    print('Number of unlinked exchanges after migration:', len(getattr(importer, 'unlinked', [])))


## 5) Write the cleaned foreground database

If unlinked exchanges are gone, the database is ready to be written to the project.


In [ ]:
if 'importer' not in globals():
    print('Run the importer cells first.')
elif len(getattr(importer, 'unlinked', [])) > 0:
    print('There are still unlinked exchanges. Inspect them before writing the database.')
else:
    if foreground_name in bd.databases:
        print(f"Database '{foreground_name}' already exists.")
    else:
        importer.write_database(foreground_name)
        print(f"Written database: {foreground_name}")


## 6) Inspect the imported foreground database

At this point, the imported foreground should be linked to BAFU and ready for analysis in the next notebook.


In [ ]:
if foreground_name in bd.databases:
    fg_db = bd.Database(foreground_name)
    print('Number of activities in foreground database:', len(fg_db))
    for act in list(fg_db)[:5]:
        print('-', act['name'], '| location =', act.get('location'), '| unit =', act.get('unit'))
else:
    print('Foreground database not written yet.')


## Troubleshooting note

If unlinked exchanges remain, do not guess blindly.
Inspect the exact names, reference products, locations, and categories first, then decide whether the fix belongs in the source file or in a migration.


## Recap

After this notebook, you should now be able to:

- import a realistic foreground Excel file
- inspect unlinked exchanges after the strategy pipeline
- distinguish technosphere and biosphere linking problems
- apply a migration file
- write a cleaned foreground database for later analysis
